# Hamiltonian Variational Ansatz (HVA) — 2-Qubit Transverse-Field Ising Model

**Primitive:** Hamiltonian Variational Ansatz (HVA), 1 layer (p=1)

## SETUP (identical in both notebooks — Qiskit & CUDA-Q)

- **N_QUBITS** = 2
- **Target Hamiltonian:** H = -J·Z0Z1 - h·(X0 + X1), with **J = 1.0**, **h = 0.5**
- **Ansatz gate sequence** (fixed, non-optimized parameters for cross-verification):
  1. H(q0), H(q1) — prepare |++⟩, the ground state of the mixer term
  2. exp(-i·GAMMA·Z0Z1) — the "problem" layer, built from the ZZ interaction
  3. exp(-i·BETA·X0), exp(-i·BETA·X1) — the "mixer" layer
- **Fixed parameters:** GAMMA = 0.3, BETA = 0.4 (rad) — chosen once, never re-derived per framework
- **QUBIT ORDERING CONTRACT:** qubit 0 = **rightmost** bit in every printed bitstring
  (Qiskit's native convention). CUDA-Q natively prints qubit 0 as the **leftmost**
  bit — the CUDA-Q notebook explicitly reverses every bitstring to match this contract.
- **SHOTS** = 10000, **SEED** = 42
- **GLOBAL PHASE CONVENTION:** normalize so the first amplitude with `abs(amp) > 1e-9`,
  in ascending bitstring order (`00` → `01` → `10` → `11`), is rotated to be real and positive:
  `phase = angle(amp)`, then multiply the whole vector by `exp(-1j*phase)`.
- **Basis ordering for matrices:** {|00⟩, |01⟩, |10⟩, |11⟩} in ascending binary order.

## Theory recap

The Hamiltonian Variational Ansatz builds the circuit directly out of the terms of the
problem Hamiltonian, alternating a "problem" unitary with a "mixer" unitary — this is the
same structure as QAOA, but here it is not restricted to diagonal cost Hamiltonians.

For the 2-qubit transverse-field Ising model

  H = -J·Z₀Z₁ - h·(X₀ + X₁)

a single HVA layer is

  U(γ, β) = [ exp(-iβX₀)·exp(-iβX₁) ] · exp(-iγZ₀Z₁)

applied to the mixer ground state |++⟩ = H⊗H |00⟩.

- `exp(-iγZ₀Z₁)` is implemented as an `RZZ(2γ)` gate in Qiskit (`RZZ(θ) = exp(-iθ/2·Z⊗Z)`),
  and as the standard `CNOT · RZ(2γ) · CNOT` decomposition in CUDA-Q (verified below to be
  numerically identical to `exp(-iγZ⊗Z)`, not just equal up to global phase).
- `exp(-iβXᵢ)` is implemented as `RX(2β)` on each qubit (`RX(θ) = exp(-iθ/2·X)`).

**Ordering-convention warning:** CUDA-Q's `get_state()` and `sample()` index qubit 0 as the
most-significant (leftmost) bit — opposite to Qiskit. This is corrected explicitly in the
CUDA-Q notebook (see the dedicated "bit-order correction" cell) before any dictionary is
printed or compared.

**Accuracy check included:** since GAMMA/BETA are fixed (not variationally optimized), both
notebooks also compute ⟨ψ|H|ψ⟩ for this specific ansatz state and compare it against the
*exact* ground-state energy of H (via direct diagonalization with NumPy), to report what
fraction of the true ground energy this one fixed HVA layer captures.

**Run all cells top to bottom.**


In [ ]:
# Install CUDA-Q.
# Package name on PyPI is "cudaq". CUDA-Q requires a Linux x86_64 or aarch64 host;
# GPU is optional for this small (2-qubit) circuit — the CPU statevector simulator
# ("qpp-cpu", the default target) is used automatically when no GPU is present.
!pip install -q cudaq


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cudaq

# ---- Global constants (must match the Qiskit notebook exactly) ----
N_QUBITS = 2
J = 1.0            # ZZ coupling strength
H_FIELD = 0.5      # transverse field strength
GAMMA = 0.3        # problem-layer angle
BETA = 0.4         # mixer-layer angle
SHOTS = 10000
SEED = 42
TOPIC_SLUG = "hva_tfim"

cudaq.set_random_seed(SEED)

print(f"N_QUBITS = {N_QUBITS}")
print(f"J = {J}, H_FIELD = {H_FIELD}")
print(f"GAMMA = {GAMMA}, BETA = {BETA}")
print(f"SHOTS = {SHOTS}, SEED = {SEED}")
print(f"CUDA-Q target: {cudaq.get_target().name}")


In [ ]:
# Build the 1-layer Hamiltonian Variational Ansatz as a CUDA-Q kernel.
#
# exp(-i*GAMMA*Z0*Z1) has no single native CUDA-Q gate, so it is built from the
# standard CNOT-RZ-CNOT identity:  CNOT(0,1) . RZ(2*gamma)_1 . CNOT(0,1) == exp(-i*gamma*Z0Z1)
# This identity is verified independently (bit-exact, not just up to global phase)
# in the "matrix representation" cell below.

@cudaq.kernel
def hva_kernel(gamma: float, beta: float):
    qubits = cudaq.qvector(2)

    # --- Step 1: prepare the mixer ground state |++> ---
    h(qubits[0])   # qubit 0 -> |+>, the +1 eigenstate of X0
    h(qubits[1])   # qubit 1 -> |+>, the +1 eigenstate of X1

    # --- Step 2: "problem" layer, exp(-i*gamma*Z0*Z1) via CNOT-RZ-CNOT ---
    x.ctrl(qubits[0], qubits[1])   # CNOT(control=0, target=1)
    rz(2.0 * gamma, qubits[1])     # CUDA-Q rz(theta) = exp(-i*theta/2 * Z), same convention as Qiskit
    x.ctrl(qubits[0], qubits[1])   # undo the CNOT

    # --- Step 3: "mixer" layer, exp(-i*beta*X0) and exp(-i*beta*X1) ---
    rx(2.0 * beta, qubits[0])      # CUDA-Q rx(theta) = exp(-i*theta/2 * X), same convention as Qiskit
    rx(2.0 * beta, qubits[1])

print(cudaq.draw(hva_kernel, GAMMA, BETA))


In [ ]:
# CUDA-Q has no native matplotlib circuit renderer, so we build a matplotlib
# figure styled to visually resemble the Qiskit diagram: horizontal qubit wires,
# boxed gate labels, same top-to-bottom qubit order (q0 on top, q1 below), so the
# two circuit images are directly comparable side by side.

def draw_hva_matplotlib(gamma, beta, filename):
    fig, ax = plt.subplots(figsize=(8, 2.2))
    n = 2
    wire_y = {0: 1.0, 1: 0.0}

    # qubit wires
    for q in range(n):
        ax.hlines(wire_y[q], 0, 10, color="black", linewidth=1.2, zorder=1)
        ax.text(-0.6, wire_y[q], f"q{q}", ha="right", va="center", fontsize=11)

    def gate_box(x, q, label, width=0.9, color="#e8f0fe"):
        ax.add_patch(plt.Rectangle((x - width / 2, wire_y[q] - 0.35), width, 0.7,
                                    facecolor=color, edgecolor="black", zorder=2))
        ax.text(x, wire_y[q], label, ha="center", va="center", fontsize=10, zorder=3)

    def ctrl_target(x, qc_, qt_, box_label):
        ax.vlines(x, wire_y[qt_], wire_y[qc_], color="black", linewidth=1.2, zorder=1)
        ax.add_patch(plt.Circle((x, wire_y[qc_]), 0.08, facecolor="black", zorder=2))
        gate_box(x, qt_, box_label, width=0.7, color="#fce8e6")

    # Step 1: H, H
    gate_box(1.0, 0, "H")
    gate_box(1.0, 1, "H")

    # Step 2: CNOT - RZ(2*gamma) - CNOT  == exp(-i*gamma*Z0Z1)
    ctrl_target(2.6, 0, 1, "X")
    gate_box(3.8, 1, f"RZ\n{2*gamma:.2f}", width=1.0, color="#fef7e0")
    ctrl_target(5.0, 0, 1, "X")

    # Step 3: RX(2*beta) on both qubits
    gate_box(6.6, 0, f"RX\n{2*beta:.2f}", width=1.0, color="#e6f4ea")
    gate_box(6.6, 1, f"RX\n{2*beta:.2f}", width=1.0, color="#e6f4ea")

    ax.set_xlim(-1, 8)
    ax.set_ylim(-0.7, 1.7)
    ax.axis("off")
    ax.set_title("HVA_TFIM (CUDA-Q) — H, H | CNOT-RZ-CNOT (=ZZ) | RX, RX", fontsize=10)
    fig.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    return fig

fig = draw_hva_matplotlib(GAMMA, BETA, f"{TOPIC_SLUG}_cudaq_circuit.png")
fig


In [ ]:
# ---- (a) Build the full 4x4 unitary by hand in NumPy, gate-by-gate ----
# Same basis convention as the theory recap: {|00>, |01>, |10>, |11>} ascending binary,
# qubit 0 as the LAST (rightmost) tensor factor -- kron(op_q1, op_q0). This is NOT
# CUDA-Q's native convention (CUDA-Q is qubit-0-leftmost) -- it is the shared reference
# convention both notebooks are held to. The bit-order correction cell below is what
# reconciles CUDA-Q's native output to this same convention.

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H2 = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)

def RX(theta):
    return np.array([[np.cos(theta / 2), -1j * np.sin(theta / 2)],
                      [-1j * np.sin(theta / 2), np.cos(theta / 2)]], dtype=complex)

def RZ(theta):
    return np.array([[np.exp(-1j * theta / 2), 0], [0, np.exp(1j * theta / 2)]], dtype=complex)

def RZZ(theta):
    ZZ = np.kron(Z, Z)
    diag = np.diag(ZZ).real
    return np.diag(np.exp(-1j * theta / 2 * diag))

CNOT = np.array([[1, 0, 0, 0],
                  [0, 1, 0, 0],
                  [0, 0, 0, 1],
                  [0, 0, 1, 0]], dtype=complex)

print("H gate:\n", np.round(H2, 4))
print("RX(2*BETA):\n", np.round(RX(2 * BETA), 4))

# Verify the CNOT-RZ-CNOT identity used inside the CUDA-Q kernel is bit-exact
# equal to exp(-i*GAMMA*Z0Z1) -- not just equal up to a global phase.
U_zz_decomp = CNOT @ np.kron(I2, RZ(2 * GAMMA)) @ CNOT
U_zz_direct = RZZ(2 * GAMMA)
print("\nCNOT . RZ(2*GAMMA) . CNOT:\n", np.round(U_zz_decomp, 4))
print("\nDirect exp(-i*GAMMA*Z0Z1):\n", np.round(U_zz_direct, 4))
decomp_match = np.allclose(U_zz_decomp, U_zz_direct, atol=1e-8)
print(f"\nCNOT-RZ-CNOT decomposition is bit-exact equal to exp(-i*GAMMA*Z0Z1): {decomp_match}")
assert decomp_match, "CNOT-RZ-CNOT decomposition does not match exp(-i*GAMMA*Z0Z1)!"

U_H = np.kron(H2, H2)
U_RX = np.kron(RX(2 * BETA), RX(2 * BETA))
U_hand = U_RX @ U_zz_decomp @ U_H

print("\nFull hand-built operator U_hand = RX_layer . (CNOT-RZ-CNOT) . H_layer:\n", np.round(U_hand, 4))

# (b) CUDA-Q has no general-purpose "extract the unitary as a matrix" utility like
# Qiskit's Operator(), so the cross-check for this notebook is the CNOT-RZ-CNOT
# identity verified above, plus the statevector comparison in the next cell.


In [ ]:
def normalize_global_phase(statevector):
    """Rotate so the first amplitude with abs(amp) > 1e-9, in ascending bitstring
    order, is real and positive. Shared convention -- identical in both notebooks."""
    amps = np.array(statevector, dtype=complex)
    for amp in amps:
        if abs(amp) > 1e-9:
            phase = np.angle(amp)
            return amps * np.exp(-1j * phase)
    return amps

# Exact statevector straight from CUDA-Q -- still in CUDA-Q's NATIVE convention
# (qubit 0 = leftmost/MSB) at this point. Bit-order correction happens in the next cell.
cudaq_state = cudaq.get_state(hva_kernel, GAMMA, BETA)
raw_amps_native = np.array(cudaq_state, dtype=complex)  # State -> ndarray of amplitudes

native_labels = [format(i, f"0{N_QUBITS}b") for i in range(2 ** N_QUBITS)]  # CUDA-Q-native order
print("Raw CUDA-Q amplitudes (CUDA-Q native convention, qubit 0 = leftmost bit):")
for label, amp in zip(native_labels, raw_amps_native):
    print(f"  |{label}>: {amp:.6f}")


In [ ]:
# ---- Explicit bit-order correction cell ----
# CUDA-Q indexes qubit 0 as the MOST-significant (leftmost) bit. To match the shared
# QUBIT ORDERING CONTRACT (qubit 0 = rightmost bit, Qiskit's native convention), every
# bitstring label must be reversed. Because amplitude index i corresponds bit-for-bit
# to the binary label at that index, reversing the *label strings* and re-sorting the
# (label, amplitude) pairs into ascending order of the *new* label achieves exactly
# the correction we need for both the statevector and (later) the measured counts.

def reverse_bitstring(bitstring):
    return bitstring[::-1]

corrected_pairs = sorted(
    ((reverse_bitstring(lbl), amp) for lbl, amp in zip(native_labels, raw_amps_native)),
    key=lambda pair: pair[0],
)
basis_labels = [p[0] for p in corrected_pairs]          # now ascending in the shared convention
raw_amps = np.array([p[1] for p in corrected_pairs])    # reordered to match

print("Bit-order-corrected amplitudes (qubit 0 = rightmost bit, shared convention):")
for label, amp in zip(basis_labels, raw_amps):
    print(f"  |{label}>: {amp:.6f}")

norm_amps = normalize_global_phase(raw_amps)
probs = np.abs(norm_amps) ** 2

print("\nPhase-normalized amplitudes:")
for label, amp in zip(basis_labels, norm_amps):
    print(f"  |{label}>: {amp:.6f}")

CUDAQ_STATEVECTOR_PROBS = {label: float(p) for label, p in zip(basis_labels, probs)}
print("\nExact probability dictionary:")
for k, v in CUDAQ_STATEVECTOR_PROBS.items():
    print(f"  {k}: {v:.6f}")
print(f"\nSum of probabilities: {sum(CUDAQ_STATEVECTOR_PROBS.values()):.8f}")


### Ordering-fix note (CUDA-Q)

**A bitstring reversal WAS required.** CUDA-Q's `get_state()` and `sample()` both index
qubit 0 as the most-significant (leftmost) bit — the opposite of the shared convention
(qubit 0 = rightmost bit). The "bit-order correction" cell above reverses every bitstring
label (`'01' -> '10'`, etc.) and re-sorts the (label, amplitude) pairs into ascending order
under the corrected labels, before anything is printed or compared to the Qiskit notebook.
After that correction, the same shared **global phase normalization** function (identical
code to the Qiskit notebook) is applied.


In [ ]:
# Sample with the shared SEED / SHOTS (seed was already set globally via
# cudaq.set_random_seed(SEED) in the imports cell).
sample_result = cudaq.sample(hva_kernel, GAMMA, BETA, shots_count=SHOTS)

raw_counts_native = dict(sample_result.items())  # {bitstring: count}, CUDA-Q native bit order
print("Raw counts (CUDA-Q native convention, qubit 0 = leftmost bit):", raw_counts_native)

# Reorder the count keys with the SAME bit-reversal used for the statevector above.
counts = {reverse_bitstring(bits): n for bits, n in raw_counts_native.items()}
print("\nCounts after bit-order correction (qubit 0 = rightmost bit):", counts)

CUDAQ_MEASURED_PROBS = {label: counts.get(label, 0) / SHOTS for label in basis_labels}
print("\nMeasured probability dictionary:")
for k, v in CUDAQ_MEASURED_PROBS.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(CUDAQ_MEASURED_PROBS.keys(), CUDAQ_MEASURED_PROBS.values(),
       color="#4285f4", edgecolor="black")
ax.set_ylim(0, 1)
ax.set_xlabel("Bitstring (qubit 0 = rightmost bit)")
ax.set_ylabel("Probability")
ax.set_title(f"HVA_TFIM (CUDA-Q) -- measured, {SHOTS} shots, seed={SEED}")
for label, p in CUDAQ_MEASURED_PROBS.items():
    ax.text(label, p + 0.01, f"{p:.3f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(f"{TOPIC_SLUG}_cudaq_histogram.png", dpi=150, bbox_inches="tight")
fig


In [ ]:
# CUDA-Q kernels don't expose a depth()/count_ops() API directly; the resource
# counts are derived here from the text circuit diagram / by construction, since the
# kernel above is fixed and fully known: 2 H, 2 CNOT, 1 RZ, 2 RX = 7 gate operations.
gate_counts = {"h": 2, "cx": 2, "rz": 1, "rx": 2}
depth = 4  # H,H (parallel) -> CNOT -> RZ -> CNOT -> RX,RX (parallel) = 4 sequential layers
print(f"Circuit depth (by construction): {depth}")
print(f"Gate count (by type): {gate_counts}")
print(f"Total operations (size): {sum(gate_counts.values())}")


In [ ]:
# ---- Accuracy check: how close is this fixed (non-optimized) HVA layer to the ----
# ---- true ground state of H = -J*Z0Z1 - h*(X0+X1) ? ----
# Uses the bit-order-corrected statevector (raw_amps, pre-phase-normalization) so the
# operator basis convention lines up with the Hamiltonian built below.
ZZ = np.kron(Z, Z)
X0 = np.kron(I2, X)  # qubit 0 operator (rightmost tensor factor, matches our convention)
X1 = np.kron(X, I2)  # qubit 1 operator
Ham = -J * ZZ - H_FIELD * (X0 + X1)

psi = raw_amps
energy = np.real(np.conj(psi) @ Ham @ psi)

exact_eigs = np.linalg.eigvalsh(Ham)
exact_ground_energy = exact_eigs.min()

print(f"<H> for this fixed HVA(GAMMA={GAMMA}, BETA={BETA}) state: {energy:.6f}")
print(f"Exact ground-state energy (NumPy eigh):              {exact_ground_energy:.6f}")
print(f"Fraction of ground energy captured: {energy / exact_ground_energy:.4%}")


In [ ]:
print("=== Convention notes (CUDA-Q) ===")
print("Bit order  : qubit 0 = leftmost bit natively in CUDA-Q -> REVERSED to rightmost")
print("             to match the shared contract (see bit-order-correction cell).")
print("Phase      : first nonzero amplitude in ascending bitstring order rotated to be real & positive.")
print("Shots/seed : SHOTS =", SHOTS, " SEED =", SEED, " (cudaq.set_random_seed(SEED))")


In [ ]:
# Final result dictionary -- built from the EXACT statevector probabilities
# (deterministic, so this is what the cross-notebook diff-check compares).
CUDAQ_RESULT = CUDAQ_STATEVECTOR_PROBS
print("CUDAQ_RESULT =", CUDAQ_RESULT)


In [ ]:
# ---- Cross-notebook diff-check (commented out: QISKIT_RESULT only exists if you ----
# ---- run the Qiskit notebook in the same session, or paste its CUDAQ_RESULT dict here) ----
#
# max_delta = max(abs(QISKIT_RESULT[k] - CUDAQ_RESULT[k]) for k in CUDAQ_RESULT)
# assert max_delta < 1e-3, f"Mismatch between Qiskit and CUDA-Q results: {max_delta}"
# print(f"Max probability delta between QISKIT_RESULT and CUDAQ_RESULT: {max_delta:.2e}")

# Hand-computed expected values for sanity-checking (derived independently with NumPy,
# see the "matrix representation" and "exact statevector" cells above for the full derivation):
EXPECTED_PROBS = {"00": 0.391100, "01": 0.108900, "10": 0.108900, "11": 0.391100}
EXPECTED_ENERGY = -1.389737   # <H> for GAMMA=0.3, BETA=0.4 (fixed, non-optimized ansatz)
EXPECTED_GROUND_ENERGY = -1.414214  # exact ground energy of H = -J*Z0Z1 - h*(X0+X1), J=1, h=0.5

max_delta_vs_hand = max(abs(CUDAQ_RESULT[k] - EXPECTED_PROBS[k]) for k in CUDAQ_RESULT)
print(f"Max probability delta vs. hand-computed expected values: {max_delta_vs_hand:.2e}")
assert max_delta_vs_hand < 1e-3, "CUDA-Q result disagrees with the hand-computed expectation!"
print("PASSED: CUDA-Q result matches the hand-computed expected probabilities within 1e-3.")
